### Boolean Information Retrieval

In [4]:
# Sample documents
docs = {
    "doc1": "information retrieval is the process of obtaining information",
    "doc2": "machine learning is related to data mining",
    "doc3": "information and data are essential in ai",
}


# Preprocessing: lowercase + split
def preprocess(text):
    return text.lower().split()


In [5]:
# Build term-document matrix (binary)
terms = {}
doc_list = list(docs.keys())

for i, (doc, text) in enumerate(docs.items()):
    words = set(preprocess(text))
    for word in words:
        if word not in terms:
            terms[word] = [0] * len(docs)
        terms[word][i] = 1


In [6]:
# Show matrix
print("\nTerm-Document Matrix:")
for term, vec in terms.items():
    print(f"{term:15}: {vec}")


# Evaluate boolean query (supports AND, OR, NOT)
def evaluate(query):
    query = query.lower().split()
    result = None
    operator = None
    i = 0

    while i < len(query):
        if query[i] == "not":
            word = query[i + 1]
            vec = terms.get(word, [0] * len(docs))
            vec = [1 - x for x in vec]
            i += 2
        elif query[i] in ["and", "or"]:
            operator = query[i]
            i += 1
            continue
        else:
            word = query[i]
            vec = terms.get(word, [0] * len(docs))
            i += 1

        if result is None:
            result = vec
        else:
            if operator == "and":
                result = [a & b for a, b in zip(result, vec)]
            elif operator == "or":
                result = [a | b for a, b in zip(result, vec)]

    return result



Term-Document Matrix:
obtaining      : [1, 0, 0]
of             : [1, 0, 0]
is             : [1, 1, 0]
process        : [1, 0, 0]
the            : [1, 0, 0]
information    : [1, 0, 1]
retrieval      : [1, 0, 0]
machine        : [0, 1, 0]
to             : [0, 1, 0]
data           : [0, 1, 1]
learning       : [0, 1, 0]
related        : [0, 1, 0]
mining         : [0, 1, 0]
and            : [0, 0, 1]
ai             : [0, 0, 1]
are            : [0, 0, 1]
in             : [0, 0, 1]
essential      : [0, 0, 1]


In [7]:
# Get user query
query = input("\nEnter Boolean query (AND, OR, NOT): ")

# Evaluate
result_vec = evaluate(query)
matches = [doc_list[i] for i, v in enumerate(result_vec) if v == 1]

# Show result
print("\nMatching Documents:", matches if matches else "No match found.")


Matching Documents: ['doc1']


### Term Weighthing Mechanism - TF, IDF, TF-IDF

In [8]:
import math

# Sample documents
documents = {
    "doc1": "information retrieval is essential",
    "doc2": "broad research is ongoing",
    "doc3": "topic modeling relates to information",
}


def preprocess(text):
    return text.lower().split()


# Build vocabulary
vocab = set()
preprocessed_docs = {}

for name, text in documents.items():
    tokens = preprocess(text)
    preprocessed_docs[name] = tokens
    vocab.update(tokens)

print(f"Preprocessed docs: {preprocessed_docs}")
vocab = sorted(list(vocab))
print("Vocabulary:", vocab)


Preprocessed docs: {'doc1': ['information', 'retrieval', 'is', 'essential'], 'doc2': ['broad', 'research', 'is', 'ongoing'], 'doc3': ['topic', 'modeling', 'relates', 'to', 'information']}
Vocabulary: ['broad', 'essential', 'information', 'is', 'modeling', 'ongoing', 'relates', 'research', 'retrieval', 'to', 'topic']


Term Frequency

In [9]:
def compute_tf(doc_tokens, vocab):
    tf = {}
    total_terms = len(doc_tokens)
    for term in vocab:
        tf[term] = doc_tokens.count(term) / total_terms
    return tf


Inverse Document Frequency

In [10]:
def compute_idf(all_docs, vocab):
    N = len(all_docs)
    idf = {}
    for term in vocab:
        df = sum(1 for doc in all_docs.values() if term in doc)
        idf[term] = math.log(N / (df + 1)) + 1  # +1 smoothing
    return idf


TF-IDF

In [11]:
def compute_tfidf(tf, idf):
    tfidf = {}
    for term in tf:
        tfidf[term] = tf[term] * idf[term]
    return tfidf


In [12]:
# Compute IDF once for all docs
idf = compute_idf(preprocessed_docs, vocab)

# For each document, compute TF and TF-IDF
for name, tokens in preprocessed_docs.items():
    tf = compute_tf(tokens, vocab)
    tfidf = compute_tfidf(tf, idf)

    print(f"\nDocument: {name}")
    print("TF:     ", {k: round(v, 3) for k, v in tf.items()})
    print("TF-IDF: ", {k: round(v, 3) for k, v in tfidf.items()})



Document: doc1
TF:      {'broad': 0.0, 'essential': 0.25, 'information': 0.25, 'is': 0.25, 'modeling': 0.0, 'ongoing': 0.0, 'relates': 0.0, 'research': 0.0, 'retrieval': 0.25, 'to': 0.0, 'topic': 0.0}
TF-IDF:  {'broad': 0.0, 'essential': 0.351, 'information': 0.25, 'is': 0.25, 'modeling': 0.0, 'ongoing': 0.0, 'relates': 0.0, 'research': 0.0, 'retrieval': 0.351, 'to': 0.0, 'topic': 0.0}

Document: doc2
TF:      {'broad': 0.25, 'essential': 0.0, 'information': 0.0, 'is': 0.25, 'modeling': 0.0, 'ongoing': 0.25, 'relates': 0.0, 'research': 0.25, 'retrieval': 0.0, 'to': 0.0, 'topic': 0.0}
TF-IDF:  {'broad': 0.351, 'essential': 0.0, 'information': 0.0, 'is': 0.25, 'modeling': 0.0, 'ongoing': 0.351, 'relates': 0.0, 'research': 0.351, 'retrieval': 0.0, 'to': 0.0, 'topic': 0.0}

Document: doc3
TF:      {'broad': 0.0, 'essential': 0.0, 'information': 0.2, 'is': 0.0, 'modeling': 0.2, 'ongoing': 0.0, 'relates': 0.2, 'research': 0.0, 'retrieval': 0.0, 'to': 0.2, 'topic': 0.2}
TF-IDF:  {'broad': 0.

## Cosine Similarity

In [13]:
# Sample documents
documents = [
    "apple banana apple dog",
    "banana banana apple dog",
]


# Build vocabulary (unique words in all documents)
def build_vocab(docs):
    vocab_set = set()
    for doc in docs:
        vocab_set.update(doc.split())
    return sorted(vocab_set)


vocab = build_vocab(documents)
print("Vocabulary:", vocab)

Vocabulary: ['apple', 'banana', 'dog']


In [14]:
# Create Bag of Words vector for a document
def bow_vector(doc, vocab):
    words = doc.split()
    return [words.count(term) for term in vocab]

In [15]:
# Compute cosine similarity between two vectors
def cosine_similarity(vec1, vec2):
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    magnitude1 = math.sqrt(sum(a * a for a in vec1))
    magnitude2 = math.sqrt(sum(b * b for b in vec2))
    if magnitude1 == 0 or magnitude2 == 0:
        return 0.0
    return dot_product / (magnitude1 * magnitude2)


# Create BoW vectors for all documents
vectors = [bow_vector(doc, vocab) for doc in documents]

# Compute similarity between first and second documents
similarity = cosine_similarity(vectors[0], vectors[1])

print("\nBoW Vector for Document 1:", vectors[0])
print("BoW Vector for Document 2:", vectors[1])
print("Cosine Similarity between Doc1 and Doc2:", round(similarity, 4))


BoW Vector for Document 1: [2, 1, 1]
BoW Vector for Document 2: [1, 2, 1]
Cosine Similarity between Doc1 and Doc2: 0.8333
